In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import sys
sys.path.append('..')

from src.problems import make_quadratic
from src.stopping_criterion import ObservableConvexCertificate
from src.penalty_method.penalties import make_classic_penalty
from src.penalty_method.optimizer import sequential_penalty_optimizer
from src.penalty_method.penalized_objective import PenalizedObjective
from src.penalty_method.projection import BallProjector

In [2]:
sigma = 0
alpha = 0.6
eps = 1
eta0 = 0.002
# gamma = 0.51
gamma = 0.51

n = 2
# x_min = np.zeros(n)
H = np.array([[2.0, 0.0], [0.0, 5.0]])
sigma = 0.1
center = np.zeros(n)
fun, true_grad, stoch_grad, proj = make_quadratic(H, sigma, center, R=np.inf)
x0 = np.r_[10, 10] * 1

In [3]:
# CONSTRAINTS + GRADIENTS
def ineq1_val(x: np.ndarray) -> float:
    # x0 + x1 - 4 <= 0
    return x[0] + x[1] - 4.0


def ineq1_grad(x: np.ndarray) -> np.ndarray:
    return np.array([1.0, 1.0])


def ineq2_val(x: np.ndarray) -> float:
    # -x <= 0
    return -x[0]


def ineq2_grad(x: np.ndarray) -> np.ndarray:
    return np.array([-1.0, 0.0])


def eq1_val(x: np.ndarray) -> float:
    # x0 -x1 - 1 = 0
    return x[0] - x[1] - 1.0


def eq1_grad(x: np.ndarray) -> np.ndarray:
    return np.array([1.0, -1.0])

In [4]:
ineq_constraints = [(ineq1_val, ineq1_grad), (ineq2_val, ineq2_grad)]
eq_constraints = [(eq1_val, eq1_grad)]

In [5]:
penalty_fun, penalty_grad = make_classic_penalty(
    inequality_constraints=ineq_constraints, equality_constraints=eq_constraints
)

In [6]:
penalized_objective = PenalizedObjective(
    base_func=fun,
    base_grad=true_grad,
    penalty_func=penalty_fun,
    penalty_grad=penalty_grad,
)

In [7]:
stopping_criterion = ObservableConvexCertificate(R=1_000, sigma=sigma, alpha=alpha, eps=eps)

In [8]:
projector = BallProjector(R=1_000, center=center)

In [11]:
x, results = sequential_penalty_optimizer(
    objective=penalized_objective,
    projector=projector.project,
    stopping_criterion=stopping_criterion,
    x0=x0,
    max_inner_iter=100_000,
    eta0=0.05,
    penalty_growth_factor=2,
)

Outer iteration 1 | Penalty multiplier: 1.0
[Iter    100]  F(x)= 0.66682 | F_avg(x)=54.48385 | G_avg=     nan | Cert=538193.47615
[Iter    200]  F(x)= 0.43709 | F_avg(x)=37.86502 | G_avg=     nan | Cert=372501.21926
[Iter    300]  F(x)= 0.41967 | F_avg(x)=30.70593 | G_avg=     nan | Cert=301277.31588
[Iter    400]  F(x)= 0.41726 | F_avg(x)=26.50014 | G_avg=     nan | Cert=259444.72185
[Iter    500]  F(x)= 0.41681 | F_avg(x)=23.65658 | G_avg=     nan | Cert=231162.29827
[Iter    600]  F(x)= 0.41671 | F_avg(x)=21.57134 | G_avg=     nan | Cert=210422.39222
[Iter    700]  F(x)= 0.41668 | F_avg(x)=19.95877 | G_avg=     nan | Cert=194383.51709
[Iter    800]  F(x)= 0.41667 | F_avg(x)=18.66403 | G_avg=     nan | Cert=181505.80986
[Iter    900]  F(x)= 0.41667 | F_avg(x)=17.59504 | G_avg=     nan | Cert=170873.29171
[Iter   1000]  F(x)= 0.41667 | F_avg(x)=16.69313 | G_avg=     nan | Cert=161902.68892
[Iter   1100]  F(x)= 0.41667 | F_avg(x)=15.91898 | G_avg=     nan | Cert=154202.72381
[Iter   12

In [12]:
x
# true [0.714, -0.286]
# f = 0.714

array([ 0.68366737, -0.27357548])